# OpenRouter Claude Sonnet 4.5 API notes
- OpenRouter exposes an OpenAI-compatible chat completions endpoint at `https://openrouter.ai/api/v1/chat/completions`.
- Authenticate with an `Authorization: Bearer {OPENROUTER_API_KEY}` header; optional `HTTP-Referer` and `X-Title` headers surface usage on OpenRouter leaderboards.
- Non-reasoning calls omit the `reasoning` object and resemble standard chat completions payloads.
- Reasoning mode for Claude models is driven via the unified `reasoning` object (e.g. `{"max_tokens": 2048}` or `{"effort": "high"}`) and returns tokens under the `reasoning` / `reasoning_details` fields while billing them as output tokens (minimum 1024, capped at 32k for Anthropic).
- The same model identifier `"anthropic/claude-sonnet-4.5"` works for both standard and reasoning-enhanced requests; switch payload parameters rather than model names.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
import requests

API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = os.environ.get("OPENROUTER_API_KEY")
if not api_key:
    raise EnvironmentError("Set OPENROUTER_API_KEY in your environment before running these examples.")

BASE_HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

DEFAULT_SYSTEM = (
    "You are Claude Sonnet 4.5 via OpenRouter. Provide concise, correct answers and include reasoning when requested."
)


In [5]:
def run_plain_completion(question: str, *, temperature: float = 0.2, max_tokens: int = 800):
    """Send a standard (non-reasoning) chat completion to Claude Sonnet 4.5.
    Returns the JSON payload from OpenRouter."""
    payload = {
        "model": "anthropic/claude-sonnet-4.5",
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": DEFAULT_SYSTEM},
            {"role": "user", "content": question},
        ],
    }
    response = requests.post(API_URL, headers=BASE_HEADERS, json=payload, timeout=60)
    response.raise_for_status()
    return response.json()

# Example usage (uncomment to execute once OPENROUTER_API_KEY is configured):
basic = run_plain_completion("Summarize the role of Aspirin in managing cardiovascular risk.")
basic_answer = basic['choices'][0]['message']['content']
print(basic_answer)


# Aspirin in Cardiovascular Risk Management

## Primary Mechanisms
- **Antiplatelet effect**: Irreversibly inhibits COX-1 enzyme, blocking thromboxane A2 production and preventing platelet aggregation
- **Reduces thrombotic events**: Decreases risk of clot formation in atherosclerotic vessels

## Secondary Prevention (Established CVD)
**Strong evidence and recommended for:**
- Prior myocardial infarction
- Prior ischemic stroke/TIA
- Stable angina or coronary artery disease
- Post-coronary revascularization (PCI/CABG)

**Benefit**: 20-25% relative risk reduction in major cardiovascular events

## Primary Prevention (No Prior CVD)
**Current approach - selective use:**
- **Age 40-59** with ≥10% 10-year CVD risk: Consider if low bleeding risk (individualized decision)
- **Age ≥60**: Generally NOT recommended due to bleeding risk outweighing benefit
- **Diabetes alone**: No longer routinely recommended

## Dosing
- **Low-dose**: 75-100 mg daily (most common)
- Higher doses don't improve ef

In [6]:
basic

{'id': 'gen-1759984129-7ZNXGoQO9RSHA9MPSc2W',
 'provider': 'Google',
 'model': 'anthropic/claude-sonnet-4.5',
 'object': 'chat.completion',
 'created': 1759984129,
 'choices': [{'logprobs': None,
   'finish_reason': 'stop',
   'native_finish_reason': 'stop',
   'index': 0,
   'message': {'role': 'assistant',
    'content': "# Aspirin in Cardiovascular Risk Management\n\n## Primary Mechanisms\n- **Antiplatelet effect**: Irreversibly inhibits COX-1 enzyme, blocking thromboxane A2 production and preventing platelet aggregation\n- **Reduces thrombotic events**: Decreases risk of clot formation in atherosclerotic vessels\n\n## Secondary Prevention (Established CVD)\n**Strong evidence and recommended for:**\n- Prior myocardial infarction\n- Prior ischemic stroke/TIA\n- Stable angina or coronary artery disease\n- Post-coronary revascularization (PCI/CABG)\n\n**Benefit**: 20-25% relative risk reduction in major cardiovascular events\n\n## Primary Prevention (No Prior CVD)\n**Current approach -

In [9]:
def run_reasoning_completion(question: str, *, reasoning_tokens: int = 8192, max_tokens: int = 90000):
    """Request Claude Sonnet 4.5 with explicit reasoning budget.
    The response includes any returned reasoning blocks under `reasoning` / `reasoning_details`."""
    payload = {
        "model": "anthropic/claude-sonnet-4.5",
        "max_tokens": max_tokens,
        "temperature": 0.2,
        "reasoning": {
            "max_tokens": reasoning_tokens,  # Anthropic minimum 1024, capped at 32000
        },
        "messages": [
            {"role": "system", "content": DEFAULT_SYSTEM + " Explain your reasoning when helpful."},
            {"role": "user", "content": question},
        ],
    }
    response = requests.post(API_URL, headers=BASE_HEADERS, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()

# Example usage (uncomment to execute once OPENROUTER_API_KEY is configured):
reasoning = run_reasoning_completion("Detail a step-by-step experimental plan to verify aspirin's antiplatelet activity.")
content = reasoning['choices'][0]['message']['content']
reasoning_trace = reasoning['choices'][0]['message'].get('reasoning')
print(content)
print(reasoning_trace)


# Experimental Plan to Verify Aspirin's Antiplatelet Activity

## In Vitro Approach (Platelet Aggregation Study)

### Materials Needed
- Fresh human blood samples (with informed consent)
- Aspirin solutions (various concentrations: 0.1-1000 μM)
- Platelet aggregometer
- Aggregation agonists (ADP, collagen, arachidonic acid)
- Anticoagulant (sodium citrate)

### Step-by-Step Protocol

**1. Sample Preparation**
- Collect blood into citrated tubes
- Prepare platelet-rich plasma (PRP) by centrifugation at 200g for 10 min
- Prepare platelet-poor plasma (PPP) by centrifuging remaining sample at 2000g for 10 min

**2. Aspirin Treatment**
- Divide PRP into control and treatment groups
- Incubate with aspirin (e.g., 100 μM) for 30 minutes at 37°C
- Include vehicle-only control

**3. Aggregation Testing**
- Add 250 μL PRP to aggregometer cuvette
- Warm to 37°C with stirring
- Calibrate: PPP = 100% light transmission, PRP = 0%
- Add agonist (e.g., 10 μM ADP or 5 μg/mL collagen)
- Record aggregati